# 🚀 Visualisierung der Einstellungsmetadaten aus den Filmen Konrad Wolfs mit *Altair*

## Festlegen der Parameter

In [ ]:
import os

# Directory and filename of the prepared metadata table.
CSV_PATH = "vis_files_shots"
CSV_FILE = "shots_meta.csv"

# Directory for the exported visualisation.
EXPORT_PATH = "export_vis_shots"

# Download the provided test files or an optional custom ZIP archive.
DOWNLOAD_TESTFILES = True
CUSTOM_ZIP_LINK = "insert link here"
CUSTOM_ARCHIVE_NAME = "custom_archive"

os.makedirs(CSV_PATH, exist_ok=True)
os.makedirs(EXPORT_PATH, exist_ok=True)

## Technische Vorbereitung

### Installation der Programmbibliotheken

In [2]:
print("Installing dependencies.")

# Install all third-party dependencies in the active notebook kernel.
# Existing packages that satisfy these requirements are reused.
%pip install -q pandas altair==6.0.0

print("Dependencies installed.")

Installing dependencies.
Note: you may need to restart the kernel to use updated packages.
Dependencies installed.


### Import der Programmbibliotheken

In [3]:
# Handling paths, opening exported charts and packaging the results.
from pathlib import Path
import webbrowser
import shutil

# Downloading files and extracting ZIP archives.
from urllib.request import urlretrieve
import zipfile

# Editing tabular data and creating interactive charts.
import pandas as pd
import altair as alt

# Displaying charts within the notebook.
from IPython.display import display

# Detect Google Colab without requiring it in a local environment.
try:
    from google.colab import files
    colab = True
except ImportError:
    files = None
    colab = False
    print("Using non-Colab environment.")

export_directory = Path(EXPORT_PATH).resolve()
alt.data_transformers.disable_max_rows()

print("Dependencies imported.")

Using non-Colab environment.
Dependencies imported.


### Download des Analysematerials

In [4]:
# Download an archive once and extract its contents into the data directory.
def download_zip(url: str, archive_name: str, target_directory: str):
    archive_path = Path(f"{archive_name}.zip")

    if not archive_path.is_file():
        print(f"Downloading {archive_path}...")
        urlretrieve(url, archive_path)
    else:
        print(f"{archive_name} already downloaded.")

    target_directory = Path(target_directory)
    with zipfile.ZipFile(archive_path, "r") as zip_ref:
        zip_ref.extractall(target_directory)
    print(f"Extracted {archive_name} to {target_directory}.")


if DOWNLOAD_TESTFILES:
    url = "https://owncloud.gwdg.de/index.php/s/YSJVSDCExtqLT01/download"
    download_zip(url, "visualisation_testfiles", CSV_PATH)
else:
    print("No testfile download requested.")

if CUSTOM_ZIP_LINK != "insert link here":
    download_zip(CUSTOM_ZIP_LINK, CUSTOM_ARCHIVE_NAME, CSV_PATH)

visualisation_testfiles already downloaded.
Extracted visualisation_testfiles to vis_files.


## Einlesen der Metadaten und Erstellung der Visualisierung

### Import und Transformation der Metadaten

In [5]:
# Load data
df = pd.read_csv(Path(CSV_PATH) / CSV_FILE)

# Pre-define metadata formats inside the df
time_metrics = [
    ("mean_s", "mean_s_format", "Mean"),
    ("median_s", "median_s_format", "Median"),
    ("max_s", "max_s_format", "Max"),
    ("min_s", "min_s_format", "Min"),
    ("runtime_s", "runtime_s_format", "Runtime"),
]

shot_metric_label = "Shot Count"

rows = []

for _, r in df.iterrows():
    title = r["title"]
    year = pd.to_numeric(r.get("year", None))

    # Time-based metrics
    for num_col, format_col, label in time_metrics:
        # Get the hh_mm_ss value
        hms = r.get(format_col, None)
        # Get the seconds value
        sec = pd.to_numeric(r.get(num_col, None), errors="coerce")
        # Get the seconds value to none if it nonexistent, otherwise transform it into a float value
        sec = None if pd.isna(sec) else float(sec)

        # Skip the row if the seconds value is empty
        if sec is None:
            continue

        # Append the row to the list of rows
        rows.append(
            {
                "title": title,
                "metric": label,
                "value": sec,
                "display_value": str(hms),
                "kind": "time",
                "time_hms": str(hms),
                "time_seconds": sec,
                "shots": None,
                "sort_value": sec,
                "release_year": year
            }
        )

    # Shot count is handled separately because it is not measured in seconds
    sc = pd.to_numeric(r.get("shot_count", None), errors="coerce")

    if pd.notna(sc):
        # The shots are saved as a float to ensure consistency within the column
        shots_int = int(round(float(sc)))
        rows.append(
            {
                "title": title,
                "metric": shot_metric_label,
                "value": float(shots_int),
                "display_value": f"{shots_int:,}",
                "kind": "count",
                "time_hms": None,
                "time_seconds": None,
                "shots": shots_int,
                "sort_value": float(shots_int),
                "release_year": year
            }
        )

df_long = pd.DataFrame(rows)

alphabetical_titles = sorted(df_long["title"].unique())

alpha_rank = {
    title: rank
    for rank, title in enumerate(alphabetical_titles)
}

df_long["alpha_rank"] = df_long["title"].map(alpha_rank)

df_long.head(14)

,title,metric,value,display_value,kind,time_hms,time_seconds,shots,sort_value,release_year,alpha_rank
0,Sonnensucher,Mean,12.0709,00:00:12,time,00:00:12,12.0709,NaN,12.0709,NaN,12
1,Sonnensucher,Median,8.4000,00:00:08,time,00:00:08,8.4000,NaN,8.4000,NaN,12
2,Sonnensucher,Max,122.2400,00:02:02,time,00:02:02,122.2400,NaN,122.2400,NaN,12
3,Sonnensucher,Min,0.0400,00:00:00,time,00:00:00,0.0400,NaN,0.0400,NaN,12
4,Sonnensucher,Runtime,6639.6400,01:50:39,time,01:50:39,6639.6400,NaN,6639.6400,NaN,12
5,Sonnensucher,Shot Count,547.0000,547,count,NaN,NaN,547.0,547.0000,NaN,12
6,Ich war neunzehn,Mean,10.6281,00:00:10,time,00:00:10,10.6281,NaN,10.6281,NaN,6
7,Ich war neunzehn,Median,6.4000,00:00:06,time,00:00:06,6.4000,NaN,6.4000,NaN,6
8,Ich war neunzehn,Max,124.7200,00:02:04,time,00:02:04,124.7200,NaN,124.7200,NaN,6
9,Ich war neunzehn,Min,0.0400,00:00:00,time,00:00:00,0.0400,NaN,0.0400,NaN,6


### Erstellung der interaktiven Visualisierung

In [6]:
# Choose the sorting order for the metrics/rows, here by extracting the third string in each of the time_metrics tuples
metric_order = [lbl for _, _, lbl in time_metrics] + [shot_metric_label]

# Assign color to each metric
color_scale = alt.Scale(
    domain=metric_order,
    range=[
        "#1f77b4",  # Mean
        "#ff7f0e",  # Median
        "#2ca02c",  # Max
        "#d62728",  # Min
        "#9467bd",  # Runtime
        "#8c564b",  # Shot Count
    ],
)

sort_mode = alt.param(
    name="sort_mode",
    value="Metric value",
    bind=alt.binding_radio(
        options=[
            "Metric value",
            "Alphabetical",
            "Chronological",
        ],
        name="Sort: ",
    ),
)

# Create a hover selection effect that…
hover = alt.selection_point(
    # …is activated when the mouse hovers over the bars
    on="mouseover", 
    # …and selects the same movie across rows
    fields=["title"],
    # …and ends when the hovering ends.
    empty=False)

# X-Axis styling
x_axis = alt.Axis(
    # Tilt film labels
    labelAngle=-35,
    # Distance between film labels 
    labelPadding=6)

#dynamic_sort = "sort_value"
# X-Axis encoding
x_enc = (
    alt.X(
        # Sets film titles as nominal categories for the bars
        "title:N",
        # Hides the titles
        title=None,
        axis=x_axis,
    )
    # Sorts the bars
    .sort(
        # Uses the sort_value from the df
        field="dynamic_sort",
        # Sort by maximum value
        op="max",
        # Sorts the bars in descending order
        order="ascending",
    )
)

base = (
    # Create the base chart from the data
    alt.Chart(df_long)
    .transform_calculate(
        dynamic_sort=(
            "sort_mode === 'Metric value' "
            "? -datum.value "
            ": sort_mode === 'Alphabetical' "
            "? datum.alpha_rank "
            ": datum.release_year"
        )
    )   

    # Define the chart's encoding
    .encode(
        # Apply the previously defined x_enc
        x=x_enc,
        # Define the y-axis encoding
        y=alt.Y(
            # Plot the entries from the value column on the y-axis as quantitative (:Q) values
            "value:Q", 
            # Don't show an y-axis title
            title=None),
        # Apply to previously defined color scheme to the metrics    
        color=alt.Color("metric:N", scale=color_scale, 
        # Hide the legend
        legend=None),
        # Define the tooltip that is displayed when hovering over a bar
        tooltip=[
            alt.Tooltip("title:N", title="Film"),
            alt.Tooltip("release_year:Q", title="Release year", format=".0f"),
            alt.Tooltip("display_value:N", title="Value"),
        ]
    )
    # Define the screen size of the chart
    .properties(width=950, height=300)
)
# Create the bars layer
bars = (
    # Turn the base into bars
    base.mark_bar()
    .encode(
        # Define hover effect styling: full opacity (1.0) on hover, otherwise (0.5)
        opacity=alt.condition(hover, alt.value(1.0), alt.value(0.5)))
        # Attach hover to bars
    .add_params(hover)
)

# Create the labels layer on top of the bars
labels = (
    alt.Chart(df_long)
    .transform_calculate(
        dynamic_sort=(
            "sort_mode === 'Metric value' "
            "? -datum.value "
            ": sort_mode === 'Alphabetical' "
            "? datum.alpha_rank "
            ": datum.release_year"
        )
    )
    # Draw the text slightly above the bar
    .mark_text(dy=-8)
    .encode(
        # Apply the same settings as above to the x-axis and y-axis to ensure that the text stays in the right place above the bar
        x=x_enc,
        y=alt.Y("value:Q"),
        # Set the color of the labels in accordance to the metric and the associated color scheme
        color=alt.Color("metric:N", scale=color_scale, legend=None),
        # Only display the label on hover
        text=alt.condition(hover, alt.Text("display_value:N"), alt.value("")),
    )
    .properties(width=950, height=300)
)

# Combine bars and labels into the final chart
chart = (
    (bars + labels)
    .add_params(sort_mode)
    .facet(
        # Create one row per metric… 
        row=alt.Row(
        # Set the facet header for each row as the metric
        "metric:N", 
        # Sort them according to the metric_order
        sort=metric_order, 
        # Hide the field title ("metric")
        title=None),
        # Set the space between rows
        spacing=15,
    )
    # Use independent y so each metric has its own range; independent x so each metric row can have its own film ordering
    .resolve_scale(x="independent", y="independent")
    # Set chart title, size and position
    .properties(title="Shot-Length Metrics + Shot Count by Film")
    .configure_title(fontSize=18, anchor="start")
    # set label and title font sizes
    .configure_axis(labelFontSize=11, titleFontSize=12)
    .configure_header(labelFontSize=14)
)
# Display the chart directly in the notebook.
display(chart)

alt.FacetChart(...)

## Export der Ergebnisse

### Export der Visualisierung als `.html`-Datei

In [7]:
# Save the chart in both local and Google Colab environments.
output_html = export_directory / "shot_meta.html"
chart.save(str(output_html))
print(f"Saved HTML chart to: {output_html}")

if not colab:
    webbrowser.open_new_tab(output_html.as_uri())

Saved HTML chart to: /Users/winter/ownCloud/quadriga/Fallstudie_2/JupyterNotebook/export/shot_meta.html


### Herunterladen des *Google Colab* Export-Ordners

In [8]:
if colab:
    # Store the ZIP archive next to the export directory, not inside it.
    archive_base = export_directory.parent / f"{export_directory.name}_results"
    zip_path = shutil.make_archive(
        str(archive_base), "zip", root_dir=str(export_directory)
    )
    files.download(zip_path)
    print(f"Downloaded {Path(zip_path).name}.")
else:
    print("Download only works in Google Colab.")

Download only works in Google Colab.
